# 1.8 Python

Python has become one of the most widely adopted languages for scientific computing and data analysis, with free distributions available across all major operating systems. The Anaconda distribution (https://www.anaconda.com) bundles Python together with all the scientific libraries used in these notes and is an easy way to get started.

At the end of each chapter we provide Python code to accompany the examples, with a particular focus on simulation. These sections assume no prior Python experience, though many free tutorials and books are available for those who want a fuller introduction.

Three libraries appear throughout:

- **NumPy** (`numpy`) — arrays and numerical operations  
- **SciPy** (`scipy`) — mathematical special functions  
- **Seaborn** (`seaborn`) — statistical visualisation

Every session begins by importing them.

In [ ]:
import math
import numpy as np
import scipy.special as sp

## Arrays

The foundation of numerical Python is the NumPy **array**. Writing code that operates on entire arrays at once — rather than looping element by element — is the key to concise, efficient scientific computing. Arrays are created with `np.array`, passing a Python list of values:

In [ ]:
v = np.array([3, 1, 4, 1, 5, 9])
v

This defines `v` as the array (3, 1, 4, 1, 5, 9). A plain assignment such as `n = 110` creates an ordinary Python integer; NumPy treats scalars and arrays uniformly in arithmetic, as we will see shortly.

Four basic summaries of an array are its total, maximum, minimum, and number of elements:

In [ ]:
print(v.sum())    # total
print(v.max())    # largest entry
print(v.min())    # smallest entry
print(len(v))     # number of entries

To build the integer sequence 1, 2, …, n, use `np.arange(1, n+1)`. The function `np.arange(a, b)` produces integers starting at `a` and stopping *before* `b`, so adding 1 to the upper bound gives an inclusive range. More generally, `np.arange(m, n+1)` yields every integer from `m` through `n`.

In [ ]:
n = 10
np.arange(1, n + 1)     # 1, 2, ..., 10

Individual elements are accessed by position. Python uses **0-based indexing**: the first element sits at index 0, the second at index 1, and so on. To retrieve the *i*-th element (counting from 1), write `v[i-1]`.

Passing a list of indices retrieves several elements at once:

In [ ]:
v = np.array([3, 1, 4, 1, 5, 9])
print(v[0])           # first element (index 0)
print(v[[0, 2, 4]])   # 1st, 3rd, and 5th elements

To drop a set of positions and keep the rest, use `np.delete` with the indices to remove:

In [ ]:
np.delete(v, [1, 2, 3])   # drop the 2nd, 3rd, and 4th entries

Arithmetic on NumPy arrays is **element-wise** by default. Raising `v` to a power, for instance, applies the exponent to each entry independently:

In [ ]:
v**3

Combining `np.arange` with element-wise operations lets us express compact mathematical sequences. For example, the array $\bigl(1,\,\tfrac{1}{4},\,\tfrac{1}{9},\,\ldots,\,\tfrac{1}{10000}\bigr)$ is just:

In [ ]:
1 / np.arange(1, 101)**2

When an operation involves an array and a scalar, NumPy **broadcasts** the scalar across every element. Adding 3 to `v`, for instance, shifts each entry by 3:

In [ ]:
v + 3

## Factorials and Binomial Coefficients

The standard library function `math.factorial(n)` computes $n!$ exactly. Python integers have unlimited precision, so `factorial` never overflows — it will return the exact value even for very large $n$. The binomial coefficient $\binom{n}{k}$ is available as `math.comb(n, k)`.

In [ ]:
print(math.factorial(10))   # 10!
print(math.comb(10, 3))     # C(10, 3)

Factorials grow so rapidly that even floating-point representations become unwieldy for large $n$. In that regime it is more practical to work with *logarithms*. `math.lgamma(n+1)` gives $\log(n!)$ (using the identity $\log(n!) = \log\,\Gamma(n+1)$), and the log of a binomial coefficient follows from the identity $\log\binom{n}{k} = \log(n!) - \log(k!) - \log((n-k)!)$:

In [ ]:
n, k = 1000, 500
print(math.lgamma(n + 1))                                                       # log(1000!)
print(sp.gammaln(n+1) - sp.gammaln(k+1) - sp.gammaln(n - k + 1))              # log C(1000, 500)

## Sampling and Simulation

NumPy's `random` module provides tools for drawing pseudo-random samples. For reproducibility we create a single **random generator** with a fixed seed and reuse it throughout:

In [ ]:
rng = np.random.default_rng(seed=42)

The method `rng.choice(population, k, replace=False)` draws an ordered sample of `k` elements from `population` without replacement, assigning equal probability to each element. For example, five numbers chosen at random from 1 through 10:

In [ ]:
n, k = 10, 5
rng.choice(np.arange(1, n + 1), k, replace=False)

Setting `replace=True` allows each element to be chosen more than once:

In [ ]:
rng.choice(np.arange(1, n + 1), k, replace=True)

`rng.permutation` generates a random ordering of an array:

In [ ]:
rng.permutation(np.arange(1, n + 1))   # random permutation of 1..n

`rng.choice` works equally well with non-numeric data. Python's `string.ascii_lowercase` contains the 26 lowercase letters; sampling 7 of them without replacement produces a random "word":

In [ ]:
import string
letters = np.array(list(string.ascii_lowercase))
''.join(rng.choice(letters, 7, replace=False))

Non-uniform probabilities are specified via the `p` argument. The call below draws three values from {1, 2, 3, 4} with replacement, where each value has the stated probability of being selected:

In [ ]:
rng.choice(np.arange(1, 5), 3, replace=True, p=[0.1, 0.2, 0.3, 0.4])

When sampling without replacement, the probability of each remaining element at every subsequent draw is rescaled to be proportional to its original weight.

Repeating a random experiment many times and recording the outcomes is called **simulation**. The standard Python idiom for this is a *list comprehension*:

```python
[expression for _ in range(N)]
```

This evaluates `expression` exactly `N` times and collects the results in a list.

## Matching Problem Simulation

Example 1.6.4 asks: if a deck of $n$ cards is shuffled at random and laid face-up one by one, what is the probability that at least one card occupies its "natural" position (card 1 in slot 1, card 2 in slot 2, and so on)? Theory tells us this probability converges to $1 - 1/e \approx 0.6321$ as $n$ grows. We can check this numerically:

In [ ]:
n = 100  # number of cards
r = [np.sum(rng.permutation(n) == np.arange(n))   # shuffle; count positions where card == slot
     for _ in range(10**4)]
np.sum(np.array(r) >= 1) / 10**4                   # fraction of trials with at least one match

Reading from the inside out:

- `rng.permutation(n)` produces a random arrangement of the integers 0 through $n-1$, representing a shuffled deck where card `i` is expected in slot `i`.
- Comparing this to `np.arange(n)` yields a Boolean array whose `i`-th entry is `True` when card `i` landed in its correct slot. `np.sum` counts those matches.
- The list comprehension repeats the experiment 10 000 times, storing the per-trial match counts in `r`.

The final line divides the number of trials that produced at least one match by the total number of trials. We can also print the result alongside the theoretical value to see how close the simulation lands:

In [ ]:
n = 100  # number of cards
r = [np.sum(rng.permutation(n) == np.arange(n))   # shuffle; count matches
     for _ in range(10**4)]
simulated = np.sum(np.array(r) >= 1) / 10**4      # proportion with at least one match

print(f'Simulated:   {simulated:.4f}')
print(f'1 - 1/e   :  {1 - 1/math.e:.4f}')

## Birthday Problem — Calculation

How likely is it that at least two people in a room of $k$ share a birthday? Assuming birthdays are uniformly distributed across 365 days, the complementary event — all birthdays distinct — has probability

$$\frac{365 \cdot 364 \cdots (365-k+1)}{365^k}.$$

`np.prod` computes the product of all elements in an array, giving a compact one-liner:

In [ ]:
k = 23
1 - np.prod(np.arange(365 - k + 1, 366)) / 365**k

It is useful to package this as a reusable function. We define `pbirthday(k)`, which returns the probability of at least one shared birthday among `k` people, and `qbirthday(p)`, which returns the minimum group size needed to achieve probability `p`. For coincidences of order higher than 2 (e.g. three people sharing a birthday) an exact closed form is harder to derive, so `pbirthday` falls back to simulation in that case.

In [ ]:
def pbirthday(k, coincident=2, classes=365):
    """P(at least `coincident` people share a birthday) with `k` people."""
    if coincident == 2:
        p_all_distinct = np.prod(np.arange(classes - k + 1, classes + 1)) / classes**k
        return 1 - p_all_distinct
    else:
        sim_rng = np.random.default_rng(0)
        trials = sim_rng.integers(0, classes, size=(50_000, k))
        max_counts = np.array([np.bincount(row).max() for row in trials])
        return float(np.mean(max_counts >= coincident))

def qbirthday(p, coincident=2, classes=365):
    """Smallest group size k such that pbirthday(k) >= p."""
    for k in range(2, classes + 1):
        if pbirthday(k, coincident=coincident, classes=classes) >= p:
            return k
    return classes

In [ ]:
print(pbirthday(23))    # ~0.507
print(qbirthday(0.5))   # 23

With 23 people the chance of a shared birthday just exceeds 50%, and 23 is indeed the smallest group size for which this holds. We can ask the same question for *triple* matches — three people sharing a birthday — by setting `coincident=3`:

In [ ]:
print(pbirthday(23, coincident=3))    # ~0.014 — only a 1.4% chance with 23 people
print(qbirthday(0.5, coincident=3))   # ~88 people needed for a 50% chance

## Birthday Problem — Simulation

To simulate the birthday problem, we generate random birthdays for 23 people and use `np.bincount` to tally how many people share each calendar day:

In [ ]:
b = rng.integers(1, 366, size=23)   # 23 birthdays drawn uniformly from days 1..365
np.bincount(b)[1:]                  # count of people born on each day

Running this many times and checking whether any day count reaches 2 gives a simulated probability:

In [ ]:
r = [np.bincount(rng.integers(0, 365, size=23)).max()   # largest birthday-count in one trial
     for _ in range(10**4)]
np.sum(np.array(r) >= 2) / 10**4                        # proportion of trials with a shared birthday

If birthdays were not equally likely across all days — for example if some months have more births than others — the exact calculation becomes considerably harder, but the simulation requires only a straightforward adjustment: pass a probability array to `rng.choice` via its `p` argument instead of using `rng.integers`.